# 02 — OCR Extraction Pipeline

**Constituency:** ชัยภูมิ เขต 2 · OCR engine: **Typhoon OCR** (`api.opentyphoon.ai`)

## Architecture: OCR-driven page dispatch

The Drive PDFs are organized **by ตำบล** — one file contains many stations stacked together. Page counts per file are not always reliable (some ตำบล have +N extra pages from re-counts), and some files mix 5/18 and 5/18 (บช) together.

Instead of mechanically splitting first then OCR\'ing each chunk, we do the opposite:

```
For each source PDF:
  1. Render every page → image
  2. OCR each page once
  3. Use the page header (หน่วยที่ X, ตำบล Y, อำเภอ Z) to identify station + form
  4. Group pages into chunks per (station × form)
  5. Parse the chunk as a unit (header + vote table)
  6. Validate (Thai-word ↔ digit, sum checks)
  7. Save to canonical schema
```

**Why this works:** each form\'s page 1 always has a clear header naming the station. We don\'t need to trust page counts.

## Output schema

### `stations.parquet` — 1 row per (station × form)
| col | type | description |
|---|---|---|
| `province`, `constituency_no` | str/int | "ชัยภูมิ", 2 |
| `form_type` | str | `5_18`, `5_18_party`, `5_16`, etc. |
| `station_code`, `station_no` | str | for 5/18 forms only |
| `district`, `subdistrict` | str | for 5/18 forms only |
| `eligible_voters`, `voters_present` | int | header counts |
| `ballots_received`, `ballots_used` | int | header counts |
| `ballots_good`, `ballots_spoiled`, `ballots_no_vote` | int | breakdown |
| `source_pdf`, `source_pages` | str | provenance |
| `ocr_status` | str | `ok` / `needs_review` / `failed` |
| `validation_flags` | str | semicolon-joined issues |

### `votes.parquet` — long format, 1 row per vote entity
| col | type | description |
|---|---|---|
| (join keys above) | | |
| `entity_kind` | str | `candidate` or `party` |
| `entity_no`, `entity_name` | int/str | from reference roster |
| `votes` | int | digit value |
| `votes_thai_word` | str | raw OCR for audit |
| `needs_review` | bool | digit ≠ word |


## 1 · Setup

In [ ]:
# !pip install pdf2image pillow opencv-python beautifulsoup4 pythainlp pandas pyarrow requests --quiet
# !apt-get install -y poppler-utils    # Linux/Colab

In [ ]:
!pip install pdf2image pillow opencv-python beautifulsoup4 pythainlp pandas pyarrow requests

In [ ]:
import os
os.environ["TYPHOON_API_KEY"] = "sk-ใส่TyphoonAPIKey ตรงนี้"

In [ ]:
import os, json, hashlib, logging
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import requests
from pypdf import PdfReader

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s | %(message)s")
log = logging.getLogger("nb01")

# Paths
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

GDRIVE_DIR     = PROJECT_ROOT / "data" / "raw" / "pdfs_gdrive"
EXTERNAL_DIR   = PROJECT_ROOT / "data" / "external"
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

# Constituency metadata
PROVINCE_NAME      = "ชัยภูมิ"
PROVINCE_CODE      = "36"
CONSTITUENCY_NO    = 2
EXPECTED_STATIONS  = 341

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF source:   {GDRIVE_DIR}")
print(f"External:     {EXTERNAL_DIR}")

In [ ]:
from pdf2image import convert_from_path
from pathlib import Path

# Test with one of your PDFs
test_pdf = next((PROJECT_ROOT / "data" / "raw" / "pdfs_gdrive").rglob("*.PDF"))
print(f"Testing: {test_pdf.name}")
pages = convert_from_path(str(test_pdf), dpi=100, first_page=1, last_page=1)
print(f"✓ pdf2image works! Page 1 size: {pages[0].size}")

In [ ]:
import os, json, re, time, base64, hashlib, logging
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
from PIL import Image
from bs4 import BeautifulSoup

try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False

try:
    from pdf2image import convert_from_path
except ImportError:
    raise ImportError("pdf2image required: pip install pdf2image && apt-get install poppler-utils")

try:
    from pythainlp.util import text_to_num
    HAS_PYTHAINLP = True
except ImportError:
    HAS_PYTHAINLP = False
    print("⚠️  pythainlp not installed — Thai number cross-check disabled")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s | %(message)s")
log = logging.getLogger("ocr")


## 2 · Configuration

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EXTERNAL_DIR   = PROJECT_ROOT / "data" / "external"
IMG_CACHE_DIR  = PROJECT_ROOT / "data" / "raw" / "images"
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
OCR_CACHE_DIR  = PROCESSED_DIR / "ocr_cache"     # 1 file per page (raw OCR text)
PAGE_INDEX_DIR = PROCESSED_DIR / "page_index"    # 1 file per source PDF (parsed headers)
STATION_OUT    = PROCESSED_DIR / "stations_raw"  # 1 JSON per (station × form)

for p in [IMG_CACHE_DIR, PROCESSED_DIR, OCR_CACHE_DIR, PAGE_INDEX_DIR, STATION_OUT]:
    p.mkdir(parents=True, exist_ok=True)

PROVINCE         = "ชัยภูมิ"
CONSTITUENCY_NO  = 2

# Typhoon OCR
TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY")
if not TYPHOON_API_KEY:
    raise RuntimeError(
        "Set TYPHOON_API_KEY in your environment first.\n"
        "  Bash:    export TYPHOON_API_KEY='sk-...'\n"
        "  Notebook: os.environ['TYPHOON_API_KEY'] = '...'  (don't commit!)"
    )

TYPHOON_URL = "https://api.opentyphoon.ai/v1/ocr"
TYPHOON_PARAMS = {
    "model": "typhoon-ocr",
    "task_type": "default",
    "max_tokens": 16384,
    "temperature": 0.1,
    "top_p": 0.6,
    "repetition_penalty": 1.2,
}

print(f"Project root:  {PROJECT_ROOT}")
print(f"OCR cache:     {OCR_CACHE_DIR}")
print(f"Page index:    {PAGE_INDEX_DIR}")
print(f"Station out:   {STATION_OUT}")


## 3 · Load reference data from nb01

In [ ]:
stations_df  = pd.read_csv(EXTERNAL_DIR / "stations.csv",  encoding="utf-8-sig")
candidates_df = pd.read_csv(EXTERNAL_DIR / "candidates.csv", encoding="utf-8-sig")
parties_df   = pd.read_csv(EXTERNAL_DIR / "parties.csv",    encoding="utf-8-sig")
manifest_df  = pd.read_csv(EXTERNAL_DIR / "source_manifest.csv", encoding="utf-8-sig")

# Build lookups
CANDIDATES = {int(r["candidate_no"]): (r["candidate_name"], r["party_name"])
              for _, r in candidates_df.iterrows()}
PARTIES = {int(r["party_no"]): r["party_name"]
           for _, r in parties_df.iterrows()}

# Tambol → list of station_codes (in order)
TAMBOL_STATIONS = {
    t: g.sort_values("station_no")["station_code"].tolist()
    for t, g in stations_df.groupby("subdistrict")
}

print(f"Loaded {len(stations_df)} stations, {len(CANDIDATES)} candidates, {len(PARTIES)} parties")
print(f"Manifest: {len(manifest_df)} (station × form) entries")
print()
print("Source kind breakdown:")
print(manifest_df["source_kind"].value_counts().to_string())


## 4 · PDF → Image preprocessing

Each PDF page → cached JPG. We crop the top header area and (if cv2 available) lightly denoise. Cache key includes DPI + crop params so re-running with different settings doesn\'t collide.


In [ ]:
def render_page(pdf_path: Path, page_idx: int, dpi: int = 200) -> Path:
    """Render one PDF page → cached JPG. Idempotent.

    Uses ASCII-only filename for the cached image (hash of source path)
    to avoid Windows/OneDrive issues with Thai characters in filenames.
    """
    # Hash includes the full source path so different PDFs don\'t collide
    key = hashlib.md5(f"{pdf_path}|{page_idx}|{dpi}".encode("utf-8")).hexdigest()[:16]
    out = IMG_CACHE_DIR / f"page_{key}.jpg"
    if out.exists():
        return out

    pages = convert_from_path(str(pdf_path), dpi=dpi,
                              first_page=page_idx + 1, last_page=page_idx + 1)
    if not pages:
        raise IndexError(f"{pdf_path.name} has no page {page_idx}")
    arr = np.array(pages[0])

    if HAS_CV2:
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        # Mild denoise — preserve thin Thai strokes
        denoised = cv2.fastNlMeansDenoising(gray, h=8)
        bgr = cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR)
        # cv2.imwrite doesn\'t handle non-ASCII paths on Windows reliably — use imencode + write
        success, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 92])
        if not success:
            raise RuntimeError(f"cv2.imencode failed for {out}")
        with open(out, "wb") as f:
            f.write(buf.tobytes())
    else:
        Image.fromarray(arr).save(out, "JPEG", quality=92)

    if not out.exists():
        raise FileNotFoundError(f"Render claimed success but {out} doesn\'t exist on disk")

    return out


def count_pdf_pages(pdf_path: Path) -> int:
    """Cheap page count without rendering."""
    from pypdf import PdfReader
    return len(PdfReader(str(pdf_path)).pages)


## 5 · Typhoon OCR client (cached, retry-safe)

In [ ]:
class TyphoonError(Exception):
    pass


def _extract_text_from_response(payload: dict) -> str:
    """
    Pull the OCR text out of Typhoon's response. Their actual shape is:
        { "results": [ { "success": True, "message": { "choices": [ {"message": {"content": "..."}} ] } } ] }

    We look at all the obvious places, in order of preference.
    """
    # Real Typhoon shape (results[0].message.choices[0].message.content)
    results = payload.get("results")
    if isinstance(results, list) and results:
        first = results[0]
        if isinstance(first, dict):
            if first.get("success") is False:
                raise TyphoonError(f"Typhoon reported failure: {first.get('error', first)}")
            msg = first.get("message", {})
            if isinstance(msg, dict):
                choices = msg.get("choices")
                if isinstance(choices, list) and choices:
                    content = choices[0].get("message", {}).get("content")
                    if content:
                        return content

    # Other shapes seen in alternative endpoints
    if "text" in payload and isinstance(payload["text"], str):
        return payload["text"]
    if "choices" in payload and isinstance(payload["choices"], list) and payload["choices"]:
        content = payload["choices"][0].get("message", {}).get("content")
        if content:
            return content

    raise TyphoonError(f"Could not extract text from payload: keys={list(payload.keys())[:10]}")


def _typhoon_request(image_path: Path, *, max_retries: int = 3, timeout: int = 60) -> str:
    """One call, with exponential backoff on transient errors."""
    headers = {"Authorization": f"Bearer {TYPHOON_API_KEY}"}
    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            with open(image_path, "rb") as f:
                resp = requests.post(
                    TYPHOON_URL,
                    headers=headers,
                    files={"file": f},
                    data=TYPHOON_PARAMS.copy(),
                    timeout=timeout,
                )

            if resp.status_code == 200:
                payload = resp.json()
                return _extract_text_from_response(payload)

            if resp.status_code in (429, 500, 502, 503, 504):
                wait = 2 ** attempt
                log.warning(f"  Typhoon HTTP {resp.status_code}, retry {attempt}/{max_retries} in {wait}s")
                time.sleep(wait)
                last_err = f"HTTP {resp.status_code}"
                continue

            raise TyphoonError(f"HTTP {resp.status_code}: {resp.text[:200]}")

        except requests.RequestException as e:
            last_err = str(e)
            time.sleep(2 ** attempt)

    raise TyphoonError(f"Exhausted retries: {last_err}")


def ocr_page(image_path: Path) -> str:
    """OCR with disk cache. Re-running this notebook skips already-OCR\'d pages."""
    cache = OCR_CACHE_DIR / f"{image_path.stem}.txt"
    if cache.exists():
        return cache.read_text(encoding="utf-8")

    raw = _typhoon_request(image_path)
    cache.write_text(raw, encoding="utf-8")
    return raw


## 6 · Header parser — what page is this?

Every page-1 of a station\'s form has a header that names:

> รายงานผลการนับคะแนน...
> หน่วยเลือกตั้งที่ **N**, หมู่ที่ M, ตำบล/แขวง/เทศบาล **TAMBOL** อำเภอ/เขต **AMPHOE**
> เขตเลือกตั้งที่ 2, จังหวัด ชัยภูมิ

Plus a form badge (top-right):
- "ส.ส. 5/18" → constituency vote
- "ส.ส. 5/18 (บช)" → party-list vote

We extract these markers from page-1 OCR. Subsequent pages (2-3 for 5/18, 2-4 for 5/18 บช) belong to the same station and just say "- 2 -", "- 3 -" etc. with no header.


In [ ]:
# Thai digit ↔ Arabic digit
THAI_NUM_MAP = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")

def thai_to_int(s) -> Optional[int]:
    """Best-effort: extract first integer from a string with Thai or Arabic digits."""
    if s is None:
        return None
    s = str(s).translate(THAI_NUM_MAP)
    m = re.search(r"\d+", s.replace(",", ""))
    return int(m.group(0)) if m else None


# Patterns
RX_FORM_PARTY     = re.compile(r"ส[\.\s]*ส[\.\s]*\s*5\s*/\s*18\s*\(\s*บช\s*\)", re.IGNORECASE)
RX_FORM_CONSTIT   = re.compile(r"ส[\.\s]*ส[\.\s]*\s*5\s*/\s*18(?!\s*\()", re.IGNORECASE)
RX_FORM_516       = re.compile(r"ส[\.\s]*ส[\.\s]*\s*5\s*/\s*16", re.IGNORECASE)
RX_FORM_517       = re.compile(r"ส[\.\s]*ส[\.\s]*\s*5\s*/\s*17", re.IGNORECASE)

# "หน่วยเลือกตั้งที่ 1" — accept Thai or Arabic digits
RX_STATION_NO = re.compile(r"หน่วยเลือกตั้งที่[\s\.]*([๐-๙\d]+)")

# "ตำบล/แขวง/เทศบาล กะฮาด" — capture word after the label
RX_TAMBOL = re.compile(r"ตำบล\s*[/]?\s*(?:แขวง\s*[/]?\s*)?(?:เทศบาล\s*)?([\u0E00-\u0E7F]+?)(?=\s+อำเภอ|\s+เขต|\s*$)")

# "อำเภอ/เขต เนินสง่า"
RX_AMPHOE = re.compile(r"อำเภอ\s*[/]?\s*(?:เขต\s+)?([\u0E00-\u0E7F]+)")

# Page numbering "- 2 -"
RX_PAGE_MARK = re.compile(r"-\s*([๐-๙\d]+)\s*-")


def detect_form_type(text: str) -> Optional[str]:
    """Identify which form this page belongs to from header text."""
    if RX_FORM_PARTY.search(text):
        return "5_18_party"
    if RX_FORM_CONSTIT.search(text):
        return "5_18"
    if RX_FORM_516.search(text):
        return "5_16_party" if "บช" in text or "บัญชีรายชื่อ" in text else "5_16"
    if RX_FORM_517.search(text):
        return "5_17_party" if "บช" in text or "บัญชีรายชื่อ" in text else "5_17"
    return None


def parse_page_header(text: str) -> dict:
    """
    Try to extract page-1 markers from OCR text.
    Returns whatever was found (some keys may be None if missing).
    """
    out = {
        "form_type":     detect_form_type(text),
        "station_no":    None,
        "tambol":        None,
        "amphoe":        None,
        "page_marker":   None,
        "is_first_page": False,
    }

    m = RX_STATION_NO.search(text)
    if m:
        out["station_no"] = thai_to_int(m.group(1))
        out["is_first_page"] = True

    m = RX_TAMBOL.search(text)
    if m:
        out["tambol"] = m.group(1).strip()

    m = RX_AMPHOE.search(text)
    if m:
        out["amphoe"] = m.group(1).strip()

    m = RX_PAGE_MARK.search(text)
    if m:
        out["page_marker"] = thai_to_int(m.group(1))

    return out


## 7 · Index pages → station chunks

For each source PDF: OCR every page once, parse header, group consecutive pages into chunks per (station × form).

Stop conditions for a chunk: next page has a different form_type OR a different station_no in its header. The "signature page" (last page of the form) is included with the previous chunk because it has no station header itself.

This step is the expensive one — call Typhoon once per page. Cached on disk so re-running is free.


In [ ]:
def index_pdf_pages(pdf_path: Path) -> list[dict]:
    """
    Walk pages of a PDF, OCR each, parse header. Returns list of page-dicts:
       {page_idx, image_path, form_type, station_no, tambol, amphoe, is_first_page, ...}
    Caches result to PAGE_INDEX_DIR/<pdf_stem>.json — re-running skips Typhoon calls.
    """
    cache = PAGE_INDEX_DIR / f"{pdf_path.stem}.json"
    if cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))

    n_pages = count_pdf_pages(pdf_path)
    pages = []
    log.info(f"Indexing {pdf_path.name} ({n_pages} pages)")

    for i in range(n_pages):
        try:
            img = render_page(pdf_path, i)
            text = ocr_page(img)
            hdr = parse_page_header(text)
            pages.append({
                "page_idx":   i,
                "image_path": str(img.relative_to(PROJECT_ROOT)),
                **hdr,
                "ocr_chars":  len(text),
            })
        except Exception as e:
            log.warning(f"  Page {i} failed: {e}")
            pages.append({"page_idx": i, "image_path": None, "form_type": None,
                          "station_no": None, "tambol": None, "amphoe": None,
                          "page_marker": None, "is_first_page": False,
                          "error": str(e)})

    cache.write_text(json.dumps(pages, ensure_ascii=False, indent=2), encoding="utf-8")
    return pages


def chunk_pages_by_station(pages: list[dict]) -> list[dict]:
    """
    Group pages into per-station chunks. A new chunk starts at every is_first_page=True.
    Pages without their own header attach to the most recent chunk.
    """
    chunks = []
    current = None

    for p in pages:
        if p.get("is_first_page"):
            if current:
                chunks.append(current)
            current = {
                "form_type":   p["form_type"],
                "station_no":  p["station_no"],
                "tambol":      p["tambol"],
                "amphoe":      p["amphoe"],
                "page_indices": [p["page_idx"]],
                "image_paths":  [p["image_path"]],
            }
        elif current:
            current["page_indices"].append(p["page_idx"])
            current["image_paths"].append(p["image_path"])
        # else: orphan page before any header — skip silently

    if current:
        chunks.append(current)

    return chunks


## 8 · Body parsers — header counts + vote table

In [ ]:
# Header field labels — robust to spacing variation
HEADER_FIELDS = {
    "eligible_voters":   r"จำนวนผู้มีสิทธิ.{0,20}ตามบัญชี",
    "voters_present":    r"(?:จำนวน)?ผู้มีสิทธิ.{0,20}มาแสดงตน",
    "ballots_received":  r"บัตรเลือกตั้งที่ได้รับ",
    "ballots_used":      r"บัตรเลือกตั้งที่ใช้",
    "ballots_good":      r"บัตรดี",
    "ballots_spoiled":   r"บัตรเสีย",
    "ballots_no_vote":   r"บัตรไม่เลือก",
}


def parse_chunk_header(combined_text: str) -> dict:
    """Extract header counts from the combined text of a chunk."""
    out = {}
    for field, label_rx in HEADER_FIELDS.items():
        # Match: <label> ... <digits> [บัตร|คน] ( <thai_word> )
        rx = re.compile(
            label_rx + r".{0,80}?([๐-๙\d,]+)\s*(?:บัตร|คน)?\s*\(([^)]{1,40})\)",
            re.DOTALL,
        )
        m = rx.search(combined_text)
        if m:
            out[field] = {"value": thai_to_int(m.group(1)),
                          "thai_word": m.group(2).strip()}
        else:
            out[field] = {"value": None, "thai_word": None}
    return out


def parse_vote_table(combined_text: str, *, kind: str) -> list[dict]:
    """
    Extract vote rows from <table> in OCR output.
    kind='candidate' -> entity_no maps to CANDIDATES dict
    kind='party'     -> entity_no maps to PARTIES dict
    """
    # Find first <table>...</table>
    m = re.search(r"<table[^>]*>(.*?)</table>", combined_text, re.DOTALL | re.IGNORECASE)
    if not m:
        return []

    table_html = m.group(0)
    soup = BeautifulSoup(table_html, "html.parser")
    rows = []

    for tr in soup.find_all("tr"):
        if tr.find("th"):  # skip header row
            continue
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(cells) < 2:
            continue

        # First cell = entity number
        entity_no = thai_to_int(cells[0])
        if entity_no is None:
            continue

        # Reference dict for valid entity_nos
        ref = CANDIDATES if kind == "candidate" else PARTIES
        if entity_no not in ref:
            continue   # spurious row

        # Find votes (digit) and Thai word in remaining cells
        votes_int = None
        thai_word = None
        for c in cells[1:]:
            n = thai_to_int(c)
            if n is not None and votes_int is None:
                votes_int = n
            elif re.search(r"[\u0E00-\u0E7F]", c):
                # text containing Thai chars and not a name/party label
                if not c.startswith(("พรรค", "นาย", "นาง", "นางสาว")):
                    thai_word = c.strip("()")

        rows.append({
            "entity_no": entity_no,
            "votes":     votes_int,
            "votes_thai_word": thai_word,
        })

    return rows


## 9 · Validation

Two consistency checks per chunk:
1. **Digit ↔ Thai word** — does `560` match `ห้าร้อยหกสิบ`?
2. **Arithmetic sums** — does `ballots_used == ballots_good + ballots_spoiled + ballots_no_vote`? Does `sum(votes) == ballots_good`?

A failed check doesn\'t drop the row — just flags it for review (these often indicate OCR errors, occasionally indicate human filling errors on the form itself, which is itself an interesting finding).

> **Note on common OCR confusables**: As you process more pages, you\'ll observe specific Thai-word OCR errors (e.g. `น`/`ม`, `ห`/`พ`, `ย`/`ธ`). Add them to `KNOWN_OCR_FIXES` below as you find them — let them grow from your own observations.


In [ ]:
# Empirically-discovered OCR fixes — start empty, grow from your own QA findings
# OCR errors we've observed in Typhoon's output for these specific forms.
# When you find new ones, add them here. The keys must match what OCR actually produces.
KNOWN_OCR_FIXES = {
    # 'X พิเศษ' / 'X ฉลาด' / 'X ปอดี' / 'X ถ้วน' — Typhoon often misreads 'ถ้วน' (final marker)
    # Usually appears after a number, ignored by word-to-int conversion anyway.
    # We strip these in normalize_thai_word() implicitly by not affecting the digit parse.
    "ปอดี": "ถ้วน",
    "พิเศษ": "ถ้วน",
    "ฉลาด": "ถ้วน",
    # 'ทำ' often misread for 'ห้า' in handwriting OCR
    "ทำร้อย": "ห้าร้อย",
}


def normalize_thai_word(w: Optional[str]) -> str:
    if not w:
        return ""
    s = w.strip().strip("()")
    for k, v in KNOWN_OCR_FIXES.items():
        s = s.replace(k, v)
    return s


def thai_word_to_int(w: Optional[str]) -> Optional[int]:
    if not w or not HAS_PYTHAINLP:
        return None
    try:
        return int(text_to_num(normalize_thai_word(w)))
    except (ValueError, KeyError, IndexError):
        return None


def cross_check(value: Optional[int], thai_word: Optional[str]) -> tuple[bool, str]:
    """Return (ok, reason). Both missing is OK (=both_missing, treated as info not error)."""
    if value is None and not thai_word:
        return True, "both_missing"
    if value is None:
        return False, "digit_missing"
    if not thai_word:
        return False, "word_missing"
    converted = thai_word_to_int(thai_word)
    if converted is None:
        return False, f"unparseable_word:{thai_word}"
    if converted != value:
        return False, f"mismatch:{value}!={converted}({thai_word})"
    return True, ""


def validate_chunk(header: dict, votes: list[dict]) -> tuple[list[str], str]:
    """Run all checks. Returns (flag_list, status)."""
    flags = []

    # Digit↔word for header fields
    for field, rec in header.items():
        ok, reason = cross_check(rec["value"], rec["thai_word"])
        if not ok:
            flags.append(f"hdr_{field}:{reason}")

    H = {k: v["value"] for k, v in header.items()}

    # ballots_used == good + spoiled + no_vote
    if all(H.get(k) is not None for k in ("ballots_used", "ballots_good", "ballots_spoiled", "ballots_no_vote")):
        s = H["ballots_good"] + H["ballots_spoiled"] + H["ballots_no_vote"]
        if s != H["ballots_used"]:
            flags.append(f"sum_used!=g+s+nv ({H['ballots_used']}!={s})")

    # sum(votes) == ballots_good
    if H.get("ballots_good") is not None:
        total = sum((v.get("votes") or 0) for v in votes)
        if total != H["ballots_good"]:
            flags.append(f"sum_votes!=good ({total}!={H['ballots_good']})")

    # Per-vote digit↔word
    for v in votes:
        ok, _ = cross_check(v.get("votes"), v.get("votes_thai_word"))
        v["needs_review"] = not ok

    status = "ok" if not flags else "needs_review"
    return flags, status


## 10 · Process one chunk → station record + vote rows

In [ ]:
def lookup_station_metadata(form_type: str, station_no: int, tambol: Optional[str]) -> dict:
    """Match (station_no, tambol) → station_code from stations.csv."""
    if form_type not in ("5_18", "5_18_party") or station_no is None:
        return {"station_code": None, "district": None, "subdistrict": None}

    # Try (station_no AND tambol) first
    if tambol:
        match = stations_df[
            (stations_df["station_no"].astype(int) == int(station_no))
            & (stations_df["subdistrict"] == tambol)
        ]
        if len(match) == 1:
            r = match.iloc[0]
            return {"station_code": r["station_code"],
                    "district":     r["district"],
                    "subdistrict":  r["subdistrict"]}

    # Fall back to (station_no only) — may be ambiguous if tambol OCR failed
    match = stations_df[stations_df["station_no"].astype(int) == int(station_no)]
    if len(match) >= 1:
        r = match.iloc[0]
        return {"station_code": r["station_code"] + ("?" if len(match) > 1 else ""),
                "district":     r["district"],
                "subdistrict":  r["subdistrict"]}

    return {"station_code": None, "district": None, "subdistrict": None}


def process_chunk(chunk: dict, source_pdf: str) -> dict:
    """
    Take an indexed chunk, fetch all its OCR'd text, parse + validate.
    Returns dict: {station: {...}, votes: [...], error: ...}
    """
    form_type = chunk["form_type"]
    if form_type is None:
        return {"error": "form_type_unknown", "station": None, "votes": []}

    # Combine OCR text from all pages of this chunk
    page_texts = []
    for img_rel in chunk["image_paths"]:
        if img_rel is None:
            continue
        img = PROJECT_ROOT / img_rel
        try:
            page_texts.append(ocr_page(img))
        except Exception as e:
            log.warning(f"  OCR retrieval failed for {img}: {e}")
    combined = "\n".join(page_texts)

    # Resolve station identity
    meta = lookup_station_metadata(form_type, chunk["station_no"], chunk["tambol"])

    # Parse body
    header = parse_chunk_header(combined)
    kind = "party" if form_type.endswith("_party") else "candidate"
    votes_raw = parse_vote_table(combined, kind=kind)
    flags, status = validate_chunk(header, votes_raw)

    station_record = {
        "province":         PROVINCE,
        "constituency_no":  CONSTITUENCY_NO,
        "form_type":        form_type,
        "station_code":     meta["station_code"],
        "station_no":       chunk["station_no"],
        "district":         meta["district"],
        "subdistrict":      meta["subdistrict"] or chunk["tambol"],
        **{k: v["value"] for k, v in header.items()},
        "source_pdf":       source_pdf,
        "source_pages":     ",".join(str(p) for p in chunk["page_indices"]),
        "ocr_status":       status,
        "validation_flags": ";".join(flags),
    }

    vote_rows = []
    ref = CANDIDATES if kind == "candidate" else PARTIES
    for v in votes_raw:
        no = v["entity_no"]
        if kind == "candidate":
            name, party = ref.get(no, ("?", "?"))
        else:
            name, party = ref.get(no, "?"), None
        vote_rows.append({
            "province":         PROVINCE,
            "constituency_no":  CONSTITUENCY_NO,
            "form_type":        form_type,
            "station_code":     meta["station_code"],
            "station_no":       chunk["station_no"],
            "district":         meta["district"],
            "subdistrict":      meta["subdistrict"] or chunk["tambol"],
            "entity_kind":      kind,
            "entity_no":        no,
            "entity_name":      name,
            "party_name":       party,
            "votes":            v.get("votes"),
            "votes_thai_word":  v.get("votes_thai_word"),
            "needs_review":     v.get("needs_review", False),
        })

    return {"station": station_record, "votes": vote_rows, "error": None}


## 11 · Batch run — process every source PDF

Iterates over unique source PDFs from the manifest. For each: index pages, chunk by station, process every chunk.

Resumable — checkpoint per (PDF + chunk) saved as JSON. Re-running picks up where it left off.

For ~30 PDFs × ~50-200 pages each → expect a few hundred Typhoon calls. Cached on disk so repeated runs skip everything that\'s already done.


In [ ]:
def chunk_checkpoint_path(source_pdf_rel: str, station_no: int, form_type: str) -> Path:
    safe = source_pdf_rel.replace("/", "__").replace("\\", "__").replace(".pdf", "").replace(".PDF", "")
    return STATION_OUT / f"{safe}__{form_type}__st{station_no}.json"


def run_batch(*, force: bool = False, limit_pdfs: Optional[int] = None,
              filter_kind: Optional[str] = None):
    """
    Process every unique source PDF in the manifest.
       force=True       — re-process even if checkpoint exists
       limit_pdfs=N     — only do first N PDFs (smoke test)
       filter_kind=...  — only PDFs of this source_kind ('pure', 'mixed', 'constituency')
    """
    # Unique source PDFs (drop missing)
    sources = manifest_df.dropna(subset=["source_pdf"]).copy()
    if filter_kind:
        sources = sources[sources["source_kind"] == filter_kind]
    unique_pdfs = sources["source_pdf"].unique()
    if limit_pdfs:
        unique_pdfs = unique_pdfs[:limit_pdfs]

    n_pdfs = len(unique_pdfs)
    n_chunks_done = 0
    n_chunks_failed = 0

    for i, src_rel in enumerate(unique_pdfs, 1):
        pdf = PROJECT_ROOT / src_rel
        log.info(f"[{i}/{n_pdfs}] {src_rel}")

        if not pdf.exists():
            log.warning(f"  Missing: {pdf}")
            continue

        try:
            pages = index_pdf_pages(pdf)
            chunks = chunk_pages_by_station(pages)
            log.info(f"  Found {len(chunks)} chunks across {len(pages)} pages")
        except Exception as e:
            log.exception(f"  Indexing failed: {e}")
            continue

        for chunk in chunks:
            if not chunk.get("station_no") or not chunk.get("form_type"):
                continue
            cp = chunk_checkpoint_path(src_rel, chunk["station_no"], chunk["form_type"])
            if cp.exists() and not force:
                continue
            try:
                result = process_chunk(chunk, src_rel)
                cp.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
                if result.get("error"):
                    n_chunks_failed += 1
                else:
                    n_chunks_done += 1
            except Exception as e:
                log.exception(f"  Chunk failed: station={chunk.get('station_no')} form={chunk.get('form_type')}")
                n_chunks_failed += 1

    log.info(f"Batch done. Processed {n_chunks_done} chunks, {n_chunks_failed} failed.")


In [ ]:
# Smoke test: 1 PDF first (uncomment when TYPHOON_API_KEY is set)
# run_batch(limit_pdfs=1)

# Full run after smoke test passes:
# run_batch()


## 12 · Aggregate checkpoints → final tables

In [ ]:
def aggregate_checkpoints():
    station_rows, vote_rows = [], []
    n_files = 0
    for cp in STATION_OUT.glob("*.json"):
        try:
            data = json.loads(cp.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            log.warning(f"Bad JSON: {cp.name}")
            continue
        n_files += 1
        if data.get("station"):
            station_rows.append(data["station"])
        vote_rows.extend(data.get("votes", []))

    log.info(f"Aggregated {n_files} checkpoint files → {len(station_rows)} station records, {len(vote_rows)} vote rows")

    stations = pd.DataFrame(station_rows)
    votes    = pd.DataFrame(vote_rows)

    # Type cleanup
    for c in ["constituency_no", "station_no", "eligible_voters", "voters_present",
              "ballots_received", "ballots_used", "ballots_good",
              "ballots_spoiled", "ballots_no_vote"]:
        if c in stations.columns:
            stations[c] = pd.to_numeric(stations[c], errors="coerce").astype("Int64")

    if "votes" in votes.columns:
        votes["votes"]     = pd.to_numeric(votes["votes"],     errors="coerce").astype("Int64")
        votes["entity_no"] = pd.to_numeric(votes["entity_no"], errors="coerce").astype("Int64")

    # Save
    stations.to_parquet(PROCESSED_DIR / "stations.parquet", index=False)
    stations.to_csv(PROCESSED_DIR / "stations.csv", index=False, encoding="utf-8-sig")
    votes.to_parquet(PROCESSED_DIR / "votes.parquet", index=False)
    votes.to_csv(PROCESSED_DIR / "votes.csv", index=False, encoding="utf-8-sig")

    print(f"Wrote stations.parquet ({len(stations)} rows), votes.parquet ({len(votes)} rows)")
    return stations, votes


# stations_df_out, votes_df_out = aggregate_checkpoints()


## 13 · QA report

In [ ]:
def qa_report():
    if not (PROCESSED_DIR / "stations.parquet").exists():
        print("Run aggregate_checkpoints() first.")
        return

    s = pd.read_parquet(PROCESSED_DIR / "stations.parquet")
    v = pd.read_parquet(PROCESSED_DIR / "votes.parquet")

    print("OCR status:")
    print(s["ocr_status"].value_counts().to_string())
    print()

    print("By form type × status:")
    print(s.groupby(["form_type", "ocr_status"]).size().unstack(fill_value=0).to_string())
    print()

    flags = s["validation_flags"].dropna().str.split(";").explode()
    flags = flags[flags.str.len() > 0]
    if len(flags):
        print(f"Top 15 validation flags ({len(flags)} total):")
        # Just take the prefix before the first ":"
        flag_types = flags.str.split(":").str[0]
        print(flag_types.value_counts().head(15).to_string())
        print()

    # Compare totals against ground truth
    ref = json.loads((EXTERNAL_DIR / "reference_constituency.json").read_text(encoding="utf-8"))
    ref_total_constit = ref["constituency"]["total_candidate_votes"]
    ref_total_party   = ref["party_list"]["total_party_votes"]

    our_total_constit = v[v["form_type"] == "5_18"]["votes"].sum()
    our_total_party   = v[v["form_type"] == "5_18_party"]["votes"].sum()

    print("=" * 60)
    print("GROUND TRUTH COMPARISON")
    print("=" * 60)
    print(f"Constituency votes:")
    print(f"  Reference (PBS):  {ref_total_constit:>10,}")
    print(f"  Our OCR:          {our_total_constit:>10,}")
    print(f"  Coverage:         {our_total_constit/ref_total_constit*100:.1f}%")
    print()
    print(f"Party-list votes:")
    print(f"  Reference (PBS):  {ref_total_party:>10,}")
    print(f"  Our OCR:          {our_total_party:>10,}")
    print(f"  Coverage:         {our_total_party/ref_total_party*100:.1f}%")
    print()

    # Save needs-review subset for manual fixing
    needs_review = s[s["ocr_status"] != "ok"]
    needs_review.to_csv(PROCESSED_DIR / "qa_needs_review.csv", index=False, encoding="utf-8-sig")
    print(f"Stations needing review: {len(needs_review)} -> qa_needs_review.csv")


# qa_report()


---

## ✅ When this notebook finishes

| Output | Description |
|---|---|
| `data/processed/ocr_cache/*.txt` | Raw OCR text per page (cached) |
| `data/processed/page_index/*.json` | Indexed pages per source PDF (cached) |
| `data/processed/stations_raw/*.json` | Per-chunk results (resumable) |
| `data/processed/stations.parquet` + `.csv` | Final station table |
| `data/processed/votes.parquet` + `.csv` | Final vote table (long format) |
| `data/processed/qa_needs_review.csv` | Rows flagged by validation |

## Recommended workflow

1. Set `TYPHOON_API_KEY` in environment
2. **Smoke test 1 PDF** — `run_batch(limit_pdfs=1)`. Inspect cached files, eyeball a JSON output. If header/votes look right, parsers are working.
3. **Tune** — common things you may need to adjust:
    - regex in `parse_chunk_header` if Typhoon formats labels differently than expected
    - `parse_vote_table` heuristics if the table column order differs
    - Add common OCR errors to `KNOWN_OCR_FIXES`
4. **Full run** — `run_batch()` (no limit). Will take ~30-60 minutes for ~30 PDFs.
5. **Aggregate** — `aggregate_checkpoints()`
6. **QA** — `qa_report()` and act on the flagged rows
7. Hand off to nb03 (cleaning) and nb04 (analysis)
